# Data Types in Python

## Learning outcomes
By the end of this notebook you should be able to:
- Compare and contrast the difference between dynamic vs static typing

## Key idea:
**Effective data-driven science and computation requires understanding how data is stored and manipulated.**

## Dynamic vs Static typing

In `Python`, data is *dynamically typed*. This means that variable types are determined and checked at **runtime**. 

In `C`, data is *statically typed*. This means that variable types are determined and checked during **compilation**.

For example, one operation in `Python` might go as follows:

```python
# Python code
result = 0
for i in range(100):
    result += i
```

The same operation in `C` would look like:

```C
/* C code */
int result = 0;
for(int i=0; i<100; i++){
    result += i;
}
```

Notice `result` was declared an `int` in `C`, whereas in `Python` it was not.

As another example, the following code will run properly in `Python`, but in `C` it raise issues.

In [ ]:
x = 4
x = "four"
x == [4]

For a Python type, there's a lot going on underneath.

## More than they seem

The standard `Python` implementation is written in `C`.

Every `Python` object is a `C` structure.

Consider an `int` for example. 
```C
int x = 42;
```

In `C`, an `int` is essentially a label for a location in memory whose bytes encode the value.

In `Python`,

```python
x = 42
x = '42'
```

would first bind the name `x` to an integer object, and then to a string object.

Looking at the `Python` 3.15.2 [source code](https://github.com/python/cpython/blob/main/Objects/clinic/longobject.c.h), an integer is encoded as a 'longobject', and the definition looks essentially like

```C
struct _longobject {       /* simplified layout */
    PyObject_HEAD          /* type pointer and bookkeeping */
    uintptr_t lv_tag;      /* sign, digit count, and flags */
    digit ob_digit[1];     /* storage for the integer's magnitude */
};
```

Therefore, `Python` integers take more data than `C` integers. 

The following illustrates the two different encodings of the integer `42`

![](imgs/c-int-vs-python-int-42.png)

Here '`PyObject_HEAD`' is a container for many things. 

We can simply see this. In `C` an `int` takes 4 bytes. In `Python`, we can use the `sys` package to check.

In [ ]:
from sys import getsizeof
print(getsizeof(42))         # Number of bytes for 42

## Lists in Python

Dynamic typing allows for *heterogeneous* lists.

In [ ]:
L = [True, 1, "one", [1], 1.0]
# [(x, type(x)) for x in L]
print(L)

In [ ]:
print(f"Number of bytes for L : {getsizeof(L)}")
[(x, type(x), getsizeof(x)) for x in L]

Consider just keeping the type the same in one list.

In [ ]:
P = [2, 3, 5, 7, 11, 13, 17, 19]
print(f"Number of bytes for P : {getsizeof(P)}")
[getsizeof(x) for x in P]

We could expect more efficiency since much of the overhead is redundant. 

One way to get better storage efficiency is to use Python's `array` module (built-in)

Since we want to do operations on arrays, `numpy` is the better approach.

## NumPy arrays

![](imgs/Numpy_logo.svg.webp)

If you want to expand upon the foundations here, check out [Numpy Fundamentals](https://numpy.org/doc/stable/user/basics.html) to learn more about, well, the fundamental ideas and philosophy present in NumPy.

Python lists contain a pointer to a block of pointers, each of which points to a full Python object (like the Python integer that we saw earlier).

Numpy arrays are fixed type and are essentially pointers to a contiguous block of data.

![](imgs/array_vs_list.png)

We import `numpy` as follows.

In [ ]:
import numpy as np

We have standards. 

We don't use anything else other than `np` for abbreviating `numpy`.

Code can get **hard** to read. By sticking to this standard, you never have to think when you see `np`.

We can construct `numpy` arrays, called `ndarray`, from Python lists. 

In [ ]:
a = np.array([2, 3, 5, 7, 11, 19])
print(type(a))
print(a)

When you feed the animal a variety of food, it makes a choice. 

In [ ]:
print(np.array([True, 1, 1.0]))
print(np.array([True, 1, 1.0, "one"]))

This is called *upcasting*. 

Can figure out the order of precedence by playing around? (You could just look it up, but that's not as fun.)

Arrays can be multi-dimensional.

In [ ]:
M = np.array([(i, i + 3, i - 1) for i in range(5) if i != 2])
print(M)

In [ ]:
T = np.array([[(i, i + 1) for i in range(4*j, 4*j + 4, 2)] for j in range(2)])
print(T)

Many ways to create arrays.

In [ ]:
# Create a length-10 integer array filled with zeros
print(np.zeros(10, dtype=int))

In [ ]:
# Create a 3x5 floating-point array filled with ones
print(np.ones((3, 5), dtype=float))

In [ ]:
# Create a 3x5 array filled with 42
print(np.full((3, 5), 42))

In [ ]:
# Create an array filled with a linear sequence. Starting at 0, ending at 20,
# stepping by 2
print(np.arange(0, 21, 2))

In [ ]:
# Create an array of five values evenly spaced between 0 and 1
print(np.linspace(0, 1, 5))

But wait, there's more!

In [ ]:
# Create a 3x4 array of continuously uniformly distributed random values between
# 0 and 1
print(np.random.random_sample((3, 4)))

In [ ]:
# Create a 3x4 array of normally distributed random values with mean 0 and
# standard deviation 1
print(np.random.normal(0, 1, (3, 4)))

In [ ]:
# Create a 2x5 array of random integers in the interval [0, 10)
print(np.random.randint(0, 10, (6, 4)))

In [ ]:
# Create a 3x3 identity matrix
print(np.eye(3, dtype=int))

## What makes NumPy arrays so special?

![](imgs/BigL.png)

[Big Lebowski clip](https://youtu.be/pWdd6_ZxX8c)

It's not a matter of opinion that `ndarrays` are fantastic. 🙃

The fundamental difference is that an `ndarray` is stored in a *homogeneous and contiguous block of memory*. 

- Computations on arrays can be written in `C`.
- Knowing the address of the memory block and the data type, it is just simple arithmetic to loop over all items.
- Spatial locality in memory access patterns results in performance gains notably due to the CPU cache.
- NumPy can take advantage of vectorized instructions of modern CPUs.

One of the downsides to this approach is if you want to add a row or column to your matrix, for example, NumPy will do an array copy. 

There are ways to avoid things like this. See [Section 4.5 of the 'Cookbook'](https://ipython-books.github.io/45-understanding-the-internals-of-numpy-to-avoid-unnecessary-array-copying/).

Two arrays may share the same memory. Be careful out there.

In [ ]:
X = np.array([1, 2, 3])
X = [1, 2, 3]
Y = X                       # Sneaky: points to same place
X[0] = 4					# Edit one
print(X, Y)					# Changes the other

### A quick note on data types within `ndarray`

One can specify the data types for an `ndarray` using the `dtype` keyword argument.

We did this above. 

It might be useful to know what some of the types mean. Here's a table:

| Data type	 | Description |
|-------------|-------------|
| `bool_`     | Boolean (True or False) stored as a byte |
| `int_`      | Default integer type (same as C `long`; normally either `int64` or `int32`)| 
| `intc`      | Identical to C `int` (normally `int32` or `int64`)| 
| `intp`      | Integer used for indexing (same as C `ssize_t`; normally either `int32` or `int64`)| 
| `int8`      | Byte (–128 to 127)| 
| `int16`     | Integer (–32768 to 32767)|
| `int32`     | Integer (–2147483648 to 2147483647)|
| `int64`     | Integer (–9223372036854775808 to 9223372036854775807)| 
| `uint8`     | Unsigned integer (0 to 255)| 
| `uint16`    | Unsigned integer (0 to 65535)| 
| `uint32`    | Unsigned integer (0 to 4294967295)| 
| `uint64`    | Unsigned integer (0 to 18446744073709551615)| 
| `float_`    | Shorthand for `float64`| 
| `float16`   | Half-precision float: sign bit, 5 bits exponent, 10 bits mantissa| 
| `float32`   | Single-precision float: sign bit, 8 bits exponent, 23 bits mantissa| 
| `float64`   | Double-precision float: sign bit, 11 bits exponent, 52 bits mantissa| 
| `complex_`  | Shorthand for `complex128`| 
| `complex64` | Complex number, represented by two 32-bit floats| 
| `complex128`| Complex number, represented by two 64-bit floats| 

## The speed of static

Let's just play around with some examples to see if NumPy is faster than Python.

We'll build an array and a list of one million entries.

![](imgs/dr_evil.jpg)

In [ ]:
numbers = np.arange(1_000_000, dtype=np.int64)		# Numpy array
as_list = numbers.tolist()							# Python list
print(type(numbers), type(as_list))

Numpy comes with home-grown ways to manipulate arrays. 

This is so that the functions themselves can take advantage of the efficiency of the arrays.

We'll come up with our own Python function, and we'll use sum.

In [ ]:
def sum_loop(values):
    total = 0
    for value in values:
        total += value
    return total

# Same numbers, same answer?
print(sum(as_list), sum_loop(as_list), numbers.sum(), sum(numbers))

Now we'll show that these fours sums are different by **orders of magnitude**!

In [ ]:
%timeit sum_loop(as_list)		# Our function
%timeit sum(as_list)			# Python's
%timeit numbers.sum()			# All Numpy
%timeit sum(numbers)			# Mixing

On an older machine of mine I got:

```
31.2 ms ± 1.99 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
6.74 ms ± 199 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
513 μs ± 16.5 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
59.1 ms ± 3.85 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
```

Compared to my MacBook Pro, three out of four runs were about 1.5-2 times slower. 

The pure NumPy computation was 5 times slower! Suggests NumPy might be able to make additional efficiencies with newer tech.

(Versions of Python and NumPy were different between the "tests".)

## Exercises

1.  Construct a NumPy $4 \times 4 \times 4$ array with all $1$ (of type `int`).

2. Construct a NumPy $7 \times 4 \times 6$ array of random integers in the range $1$ to $99$ (inclusive).
   
3. Create an array of $71$ values, evenly spaced between $0$ and $100$.

4. Create an array with a sequence of integers, starting at $1950$, ending at $2015$, stepping by $5$.
   
5. Create a list of all odd squares between $0$ and $10000$.

6. Make a few NumPy arrays, with random entries or ranges of integers, of varying dimensions.

7. Determine the basic attributes of these arrays.